In [ ]:
import time
import math


class WorldModel:

    def __init__(self):

        # object database
        self.objects = {}

        # maximum objects stored
        self.max_objects = 50


    # --------------------------------
    # Update world memory
    # --------------------------------
    def update(self, detections):

        now = time.time()

        for obj in detections:

            obj_id = obj["id"]

            if obj_id not in self.objects:

                # new object
                self.objects[obj_id] = {
                    "id": obj_id,
                    "object": obj["object"],
                    "position": obj["position"],
                    "confidence": obj["confidence"],
                    "stable_frames": obj.get("stable_frames", 0),
                    "first_seen": now,
                    "last_seen": now,
                    "timestamp": now
                }

            else:

                existing = self.objects[obj_id]

                # update fields
                existing["position"] = obj["position"]
                existing["confidence"] = max(existing["confidence"], obj["confidence"])
                existing["stable_frames"] = obj.get("stable_frames", existing["stable_frames"])
                existing["last_seen"] = now
                existing["timestamp"] = now


        # prevent memory overflow
        if len(self.objects) > self.max_objects:
            self.cleanup_oldest()


    # --------------------------------
    # Remove stale objects
    # --------------------------------
    def decay(self):

        now = time.time()

        for obj_id in list(self.objects.keys()):

            obj = self.objects[obj_id]

            elapsed = now - obj["last_seen"]

            # reduce confidence gradually
            if elapsed > 5:
                obj["confidence"] *= 0.9

            # delete stale objects
            if elapsed > 10 or obj["confidence"] < 0.3:
                del self.objects[obj_id]


    # --------------------------------
    # Remove oldest objects if memory full
    # --------------------------------
    def cleanup_oldest(self):

        sorted_objs = sorted(
            self.objects.items(),
            key=lambda x: x[1]["last_seen"]
        )

        # remove oldest
        while len(self.objects) > self.max_objects:
            oldest_id = sorted_objs[0][0]
            del self.objects[oldest_id]
            sorted_objs.pop(0)


    # --------------------------------
    # Get all objects
    # --------------------------------
    def get_all(self):

        return list(self.objects.values())


    # --------------------------------
    # Get objects by label
    # --------------------------------
    def get_by_label(self, label):

        return [
            obj for obj in self.objects.values()
            if obj["object"] == label
        ]


    # --------------------------------
    # Scene relationship detection
    # --------------------------------
    def scene_relations(self):

        relations = []

        objs = list(self.objects.values())

        for i in range(len(objs)):
            for j in range(i + 1, len(objs)):

                p1 = objs[i]["position"]
                p2 = objs[j]["position"]

                dist = math.sqrt(
                    (p1[0] - p2[0])**2 +
                    (p1[1] - p2[1])**2 +
                    (p1[2] - p2[2])**2
                )

                if dist < 0.5:

                    relations.append(
                        f"{objs[i]['object']} near {objs[j]['object']}"
                    )

        return relations


    # --------------------------------
    # Debug print
    # --------------------------------
    def debug(self):

        print("\n--- WORLD MODEL ---")

        for obj in self.objects.values():

            age = time.time() - obj["first_seen"]

            print(
                f"{obj['object']} | ID:{obj['id']} | "
                f"dist:{obj['position'][2]:.2f}m | "
                f"conf:{obj['confidence']:.2f} | "
                f"age:{age:.1f}s"
            )

        print("-------------------\n")